# F1 Race Outcome Predictor

This notebook documents the research and modelling behind the F1 Race Outcome Predictor — a **LightGBM learning-to-rank model** trained on 8 seasons of race data (2018–2026).

The key framing decision: we predict *relative order* within a race, not raw finish numbers. A regression model that predicts Norris finishes 3rd and Verstappen finishes 2.9th gets the order wrong even though the numbers are close. A learning-to-rank model is penalised directly for getting the ordering wrong.

**Sections:**
1. Dataset Overview
2. EDA — What predicts race outcome?
3. Feature Engineering
4. Model: LGBMRanker
5. SHAP Feature Importance
6. Ablation Study
7. Live Prediction — Next Upcoming GP

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make src/ importable when launching Jupyter from the project root
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
from src.data.cleaner import build_base_dataframe
from src.features.engineer import prepare_features, finish_to_relevance

df = build_base_dataframe()
print(f"Rows: {len(df):,}  |  Races: {df['date'].nunique()}  |  Seasons: {df['year'].min()}–{df['year'].max()}")

## 1. Dataset Overview

The dataset merges the Ergast F1 database (historical CSVs) with FastF1 (live results for any race after the Ergast export cutoff). Every row is one driver in one race.

In [ ]:
# Races per season — shows coverage and the grid expansion to 22 cars in 2026
races_per_year = df.groupby("year")["date"].nunique()

fig, ax = plt.subplots(figsize=(10, 3.5))
races_per_year.plot.bar(ax=ax, color="#e10600", edgecolor="white", width=0.7)
ax.set_xlabel("Season")
ax.set_ylabel("Races")
ax.set_title("Races per Season in Training Data")
ax.bar_label(ax.containers[0], padding=2, fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

if missing.empty:
    print("No missing values after cleaning.")
else:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing) * 0.4)))
    missing.plot.barh(ax=ax, color="#3671C6")
    ax.set_title("Missing Values by Column")
    ax.set_xlabel("Count")
    plt.tight_layout()
    plt.show()

print(df.dtypes.to_string())

## 2. EDA — What Predicts Race Outcome?

Before building features, let's look at what the data actually shows about race outcomes.

### Grid Position vs Finish

Qualifying is the strongest single predictor. The chart below splits by rain — wet races show much higher variance (starts from the back matter less; reliability matters more).

In [ ]:
finished = df[(df["driver_dnf"] == 0) & (df["team_dnf"] == 0) & df["finish"].between(1, 20)].copy()
finished["weather"] = finished["rainfall"].map({0: "Dry", 1: "Wet"})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
palette = {"Dry": "#3671C6", "Wet": "#27F4D2"}

for ax, (label, grp) in zip(axes, finished.groupby("weather")):
    ax.scatter(grp["start"], grp["finish"], alpha=0.15, s=8, color=palette[label])
    z = np.polyfit(grp["start"], grp["finish"], 1)
    xs = np.linspace(1, 20, 100)
    ax.plot(xs, np.poly1d(z)(xs), color="black", linewidth=1.5)
    ax.set_title(f"{label} races  (n={len(grp):,})")
    ax.set_xlabel("Grid position (start)")
    ax.set_ylabel("Finish position")
    ax.invert_yaxis()
    corr = grp[["start", "finish"]].corr().iloc[0, 1]
    ax.text(0.05, 0.95, f"r = {corr:.2f}", transform=ax.transAxes, va="top", fontsize=10)

plt.suptitle("Grid → Finish: stronger in dry conditions", y=1.02)
plt.tight_layout()
plt.show()

### Team Reliability (DNF Rate)

Mechanical failures are strongly team-dependent. The chart shows each team's average DNF rate over the dataset — this is what motivates the `team_reliability_ewm10` feature.

In [ ]:
from src.config import ACTIVE_CONSTRUCTORS

team_dnf = (
    df[df["team"].isin(ACTIVE_CONSTRUCTORS)]
    .groupby("team")["team_dnf"]
    .mean()
    .sort_values(ascending=True)
    .mul(100)
)

TEAM_COLOURS = {
    "McLaren": "#FF8000", "Ferrari": "#DC0000", "Red Bull": "#3671C6",
    "Mercedes": "#27F4D2", "Aston Martin": "#358C75", "Alpine": "#FF87BC",
    "Williams": "#64C4FF", "Racing Bulls": "#6692FF", "Haas F1 Team": "#B6BABD",
    "Audi": "#E8002D", "Cadillac": "#003087",
}
colours = [TEAM_COLOURS.get(t, "#888") for t in team_dnf.index]

fig, ax = plt.subplots(figsize=(8, 5))
team_dnf.plot.barh(ax=ax, color=colours, edgecolor="none")
ax.set_xlabel("Team DNF rate (%)")
ax.set_title("Mechanical DNF Rate by Team (2018–2026)")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

### Circuit Affinity: Does Track History Matter?

Some drivers consistently outperform at specific circuits. The chart below compares top drivers at circuits with very different characteristics — this motivates the `driver_circuit_avg` feature.

In [ ]:
focus_drivers  = ["Verstappen", "Norris", "Leclerc", "Hamilton"]
focus_circuits = [
    "Circuit de Monaco",
    "Autodromo Nazionale di Monza",
    "Silverstone Circuit",
    "Circuit de Barcelona-Catalunya",
]

affinity = (
    df[
        df["driver"].isin(focus_drivers) &
        df["circuit name"].isin(focus_circuits) &
        (df["driver_dnf"] == 0) & (df["team_dnf"] == 0)
    ]
    .groupby(["driver", "circuit name"])["finish"]
    .mean()
    .reset_index()
    .pivot(index="circuit name", columns="driver", values="finish")
)

fig, ax = plt.subplots(figsize=(9, 4))
affinity.plot.bar(ax=ax, width=0.7)
ax.set_ylabel("Mean finish position (lower = better)")
ax.set_title("Average Finish by Driver and Circuit (DNFs excluded)")
ax.invert_yaxis()
ax.legend(title="Driver", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 3. Feature Engineering

All rolling features use `shift(1)` — a race never sees its own result in its lookback window. This prevents data leakage: when predicting race N, only results from races 1..N-1 are visible.

| Feature | What it captures |
|---------|------------------|
| `start` | Qualifying grid position |
| `finish_ewm3` | Exponentially-weighted recent form (span=3) |
| `driver_reliability_ewm10` | 1 − EWM driver DNF rate (span=10) |
| `team_reliability_ewm10` | 1 − EWM team DNF rate (span=10) |
| `driver_circuit_avg` | Expanding historical average finish at this circuit |
| `age_at_race` | Driver age on race day |
| `rainfall` | Binary rain flag |
| `avg_track_temp` | Median track temp (°C) |
| `min_humidity` | Minimum humidity (%) |
| `driver_active` / `team_active` | Current-grid flags |
| `GP_name`, `team`, `driver`, `circuit_name` | Categorical identities (native LightGBM) |

In [ ]:
X, y = prepare_features(df)

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
print("\nNumerical features:")
print(X.select_dtypes(exclude="category").describe().round(2).T)
print("\nCategorical features:", [c for c in X.columns if X[c].dtype.name == "category"])

## 4. Model: LGBMRanker (Learning-to-Rank)

**Why LambdaRank instead of regression?**

A regression model minimises prediction error on raw positions — but a model that predicts [2.1, 2.0, 3.5] still gets the top-2 order wrong, even if the MSE is small. LambdaRank directly optimises the ranking within each race (rows are grouped by race date). It penalises mistakes at the front of the field more than at the back, which matches real usage — getting the podium right matters more than getting P18 vs P19 right.

Hyperparameters were found via 30-trial Optuna search with NDCG@10 as the objective, using time-series cross-validation (no future data leaks into fold validation).

In [ ]:
from src.config import MODEL_PATH, TEST_FRAC
from src.models.train import BEST_PARAMS, fit
from src.models.evaluate import summarize

print("Best hyperparameters (Optuna, 30 trials, NDCG@10 objective):")
for k, v in BEST_PARAMS.items():
    print(f"  {k}: {v}")

In [ ]:
# Chronological 80/20 split — the most recent races form the test set
dates  = sorted(df["date"].unique())
cutoff = dates[int(len(dates) * (1 - TEST_FRAC))]
print(f"Training through: {str(dates[int(len(dates)*(1-TEST_FRAC))-1])[:10]}")
print(f"Test set starts:  {str(cutoff)[:10]}")
print(f"Training races: {sum(d < cutoff for d in dates)}  |  Test races: {sum(d >= cutoff for d in dates)}")

train_df = df[df["date"] < cutoff]
test_df  = df[df["date"] >= cutoff]

model, feature_names = fit(train_df)
print(f"\nModel trained on {len(feature_names)} features.")

In [ ]:
X_test, y_test = prepare_features(test_df)
pred_df = test_df.sort_values(["date", "driver"]).reset_index(drop=True).copy()
pred_df["pred_score"] = model.predict(X_test[feature_names])
pred_df["pred_finish"] = (
    pred_df.groupby("date")["pred_score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

metrics = summarize(pred_df)
print("\nTest-set evaluation (chronological hold-out — no data leakage):\n")
print(f"  Mean Spearman rank correlation : {metrics['mean_spearman']:.3f}")
print(f"  Median Spearman                : {metrics['median_spearman']:.3f}")
print(f"  NDCG@3  (podium accuracy)      : {metrics['ndcg_3']:.3f}")
print(f"  NDCG@10 (points-finish)        : {metrics['ndcg_10']:.3f}")
print(f"  NDCG@20 (full field)           : {metrics['ndcg_20']:.3f}")

## 5. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature contributions. The beeswarm plot shows both *which features matter most* (y-axis) and *in which direction* (colour: red = high feature value, blue = low).

In [ ]:
import shap

X_sample = X_test[feature_names].sample(min(2000, len(X_test)), random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_sample, plot_type="beeswarm", show=False, max_display=15)
plt.title("SHAP Feature Importance (beeswarm)")
plt.tight_layout()
plt.show()

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names,
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 5))
mean_abs_shap.plot.barh(ax=ax, color="#e10600", edgecolor="none")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Top-15 Features by Mean Absolute SHAP")
plt.tight_layout()
plt.show()

## 6. Ablation Study

We tested three modifications on top of the baseline (flat 5-race rolling window, default LightGBM params). Results are from the same chronological hold-out split.

**Key findings:**
- EWM features + Optuna tuning gave the biggest lift — recent form weighted more than older races, and the tuned tree depth/learning rate matters
- Adding a DNF classifier (XGBoost binary, predicting retirement before the race) hurt performance. The features available pre-race don't reliably predict mechanical failures, and incorrect DNF predictions pollute the ranker's input
- Ensembling the ranker with the DNF classifier (averaging per-race ranks) recovered some Spearman but traded podium accuracy (NDCG@3) for mid-field accuracy — the wrong tradeoff for race prediction

In [ ]:
ablation = pd.DataFrame([
    {"Configuration": "Baseline (flat 5-race rolling, default params)",
     "Mean Spearman": 0.655, "NDCG@3": 0.875, "NDCG@10": 0.875, "NDCG@20": 0.945},
    {"Configuration": "+ EWM features + Optuna tuning  ← final model",
     "Mean Spearman": 0.670, "NDCG@3": 0.910, "NDCG@10": 0.892, "NDCG@20": 0.954},
    {"Configuration": "+ DNF classifier (XGBoost)",
     "Mean Spearman": 0.665, "NDCG@3": 0.900, "NDCG@10": 0.882, "NDCG@20": 0.950},
    {"Configuration": "+ Ensemble (ranker + DNF, averaged ranks)",
     "Mean Spearman": 0.671, "NDCG@3": 0.887, "NDCG@10": 0.880, "NDCG@20": 0.951},
])

def _highlight_best(s):
    if s.name == "Configuration":
        return [""] * len(s)
    is_max = s == s.max()
    return ["background-color: #d4edda; font-weight: bold" if v else "" for v in is_max]

ablation.style.apply(_highlight_best).format(
    {"Mean Spearman": "{:.3f}", "NDCG@3": "{:.3f}", "NDCG@10": "{:.3f}", "NDCG@20": "{:.3f}"}
).hide(axis="index")

## 7. Live Prediction — Next Upcoming GP

The predictor loads the feature snapshot saved at training time (each driver's latest rolling reliability and form), builds feature rows for the upcoming race using the current grid + weather inputs, and ranks all drivers in one forward pass.

Re-run this cell before each race weekend to get the latest prediction.

In [ ]:
import fastf1
from datetime import date
from difflib import get_close_matches
from src.config import CACHE_DIR

CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

KNOWN_RACES = [
    "Australian Grand Prix", "Chinese Grand Prix", "Japanese Grand Prix",
    "Bahrain Grand Prix", "Saudi Arabian Grand Prix", "Miami Grand Prix",
    "Emilia Romagna Grand Prix", "Monaco Grand Prix", "Spanish Grand Prix",
    "Canadian Grand Prix", "Austrian Grand Prix", "British Grand Prix",
    "Belgian Grand Prix", "Hungarian Grand Prix", "Dutch Grand Prix",
    "Italian Grand Prix", "Azerbaijan Grand Prix", "Singapore Grand Prix",
    "United States Grand Prix", "Mexico City Grand Prix", "São Paulo Grand Prix",
    "Las Vegas Grand Prix", "Qatar Grand Prix", "Abu Dhabi Grand Prix",
]

today = pd.Timestamp(date.today())
race_name, race_date = "Canadian Grand Prix", pd.Timestamp("2026-06-14")  # fallback

for year in [today.year, today.year + 1]:
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        upcoming = schedule[pd.to_datetime(schedule["EventDate"]) > today]
        if not upcoming.empty:
            row = upcoming.iloc[0]
            name = str(row["EventName"])
            if name not in KNOWN_RACES:
                matches = get_close_matches(name, KNOWN_RACES, n=1, cutoff=0.6)
                name = matches[0] if matches else KNOWN_RACES[0]
            race_name, race_date = name, pd.Timestamp(row["EventDate"])
            break
    except Exception:
        continue

print(f"Next race: {race_name}  |  Date: {race_date.date()}")

In [ ]:
from src.predict.predictor import F1Predictor
from src.predict.prep import build_race_features

DRIVER_TEAMS = {
    "Antonelli": "Mercedes",   "Russell": "Mercedes",
    "Norris": "McLaren",       "Piastri": "McLaren",
    "Leclerc": "Ferrari",      "Hamilton": "Ferrari",
    "Verstappen": "Red Bull",  "Hadjar": "Red Bull",
    "Alonso": "Aston Martin",  "Stroll": "Aston Martin",
    "Gasly": "Alpine",         "Colapinto": "Alpine",
    "Sainz": "Williams",       "Albon": "Williams",
    "Lawson": "Racing Bulls",  "Lindblad": "Racing Bulls",
    "Bearman": "Haas F1 Team", "Ocon": "Haas F1 Team",
    "Hulkenberg": "Audi",      "Bortoleto": "Audi",
    "Perez": "Cadillac",       "Bottas": "Cadillac",
}

grid = [
    {"driver": drv, "team": team, "start": i + 1}
    for i, (drv, team) in enumerate(DRIVER_TEAMS.items())
]

# Update these before each race
rainfall       = 0
avg_track_temp = 35.0
min_humidity   = 40.0

predictor = F1Predictor()
race_features = build_race_features(
    race_name=race_name,
    race_date=str(race_date.date()),
    grid=grid,
    rainfall=rainfall,
    avg_track_temp=avg_track_temp,
    min_humidity=min_humidity,
)
for col in predictor.feature_names:
    if col not in race_features.columns:
        race_features[col] = 0.5

result = predictor.predict(race_features)

scores = result["pred_score"].values
exp_s  = np.exp(scores - scores.max())
result["confidence"] = exp_s / exp_s.sum() * 100

out = result[["pred_finish", "driver", "team", "confidence"]].copy()
out.columns = ["Predicted Finish", "Driver", "Team", "Confidence %"]
out["Confidence %"] = out["Confidence %"].map("{:.1f}%".format)

def colour_team(val):
    c = TEAM_COLOURS.get(val, "#333")
    return f"color: {c}; font-weight: bold"

out.style.applymap(colour_team, subset=["Team"]).hide(axis="index")